In [0]:
train_path = "/Volumes/workspace/default/cmapss_data/train_FD001.txt"

train_df = spark.read \
    .option("sep", " ") \
    .option("inferSchema", "true") \
    .csv(train_path)

display(train_df.limit(10))

In [0]:
print("Rows:", train_df.count())
print("Columns:", len(train_df.columns))

In [0]:
train_df = train_df.drop("_c26", "_c27")

print("Rows:", train_df.count())
print("Columns:", len(train_df.columns))

In [0]:
columns = [
    "unit_number",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_5",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_16",
    "sensor_17",
    "sensor_18",
    "sensor_19",
    "sensor_20",
    "sensor_21"
]

train_df = train_df.toDF(*columns)

display(train_df.limit(10))

In [0]:
print(train_df.columns)

In [0]:
from pyspark.sql.functions import col, sum

missing_values = train_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in train_df.columns
])

display(missing_values)

In [0]:
duplicate_count = train_df.count() - train_df.dropDuplicates().count()

print("Duplicate rows:", duplicate_count)

In [0]:
print("Number of engines:", train_df.select("unit_number").distinct().count())

In [0]:
train_df.groupBy("unit_number") \
    .agg({"cycle": "max"}) \
    .orderBy("unit_number") \
    .show(100)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import max as spark_max, col

window_engine = Window.partitionBy("unit_number")

train_df = train_df.withColumn(
    "max_cycle",
    spark_max("cycle").over(window_engine)
)

train_df = train_df.withColumn(
    "RUL",
    col("max_cycle") - col("cycle")
)

display(train_df.limit(10))

In [0]:
train_df = train_df.drop("max_cycle")

print("Rows:", train_df.count())
print("Columns:", len(train_df.columns))

display(train_df.limit(5))

In [0]:
train_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_cmapss_train")

In [0]:
display(spark.table("workspace.default.silver_cmapss_train"))

In [0]:
test_path = "/Volumes/workspace/default/cmapss_data/test_FD001.txt"

test_df = spark.read \
    .option("sep", " ") \
    .option("inferSchema", "true") \
    .csv(test_path)

print("Rows:", test_df.count())
print("Columns:", len(test_df.columns))

In [0]:
test_df = test_df.drop("_c26", "_c27")

print("Rows:", test_df.count())
print("Columns:", len(test_df.columns))

In [0]:
test_df = test_df.toDF(*columns)

display(test_df.limit(10))

In [0]:
columns = [
    "unit_number",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_5",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_16",
    "sensor_17",
    "sensor_18",
    "sensor_19",
    "sensor_20",
    "sensor_21"
]

test_df = test_df.toDF(*columns)

display(test_df.limit(10))

In [0]:
missing_test = test_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in test_df.columns
])

display(missing_test)

In [0]:
from pyspark.sql.functions import col, sum

missing_test = test_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in test_df.columns
])

display(missing_test)

In [0]:
duplicate_test = test_df.count() - test_df.dropDuplicates().count()

print("Duplicate rows:", duplicate_test)

In [0]:
print("Number of test engines:", test_df.select("unit_number").distinct().count())

In [0]:
test_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_cmapss_test")

In [0]:
rul_path = "/Volumes/workspace/default/cmapss_data/RUL_FD001.txt"

rul_df = spark.read \
    .option("inferSchema", "true") \
    .csv(rul_path)

print("Rows:", rul_df.count())
print("Columns:", len(rul_df.columns))

display(rul_df.limit(10))

In [0]:
rul_df = rul_df.withColumnRenamed("_c0", "RUL")

display(rul_df.limit(10))

In [0]:
print("Rows:", rul_df.count())
print("Columns:", len(rul_df.columns))

rul_df.select("RUL").summary().show()

In [0]:
rul_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_cmapss_rul")